# Build a RAG Agent Lab

<div class="alert alert-block alert-info">
    <b>Note:</b> 
    <p>
        This lab is a direct copy of the
        <a href="https://mastra.ai/en/guides/guide/research-assistant">Mastra Research Assistant Guide</a> 
        I have only expanded upon these concepts with my own thoughts and references.
    </p>
    <p>
        Before starting this lab you will need an OpenAi API Key and a Postgres Connection String.
    </p>
</div>

This lab will show you how to use Mastra to create a Retrieval-Augmented Generation (RAG) system. These systems 
incorporate resource retrieval mechanisms that allow the Ai to gain more context by querying external sources like
the internet or internal tooling. In it you will learn how to provide a text based document to a Vector Store and
query it.

In Jupyter notebooks (including those running JavaScript or TypeScript via tslab), each code cell is often executed in its own scope. This means that variables defined in one cell may not be accessible in another, which can be frustrating when building up a project interactively.
 
To address this, you can use the `globalThis` object. `globalThis` is a standard JavaScript object that provides a universal way to access the global scope, regardless of the environment (browser, Node.js, etc.). By attaching variables to `globalThis`, you ensure they persist and are accessible across all cells in your notebook.
 
**Example:**
 
```typescript
// In one cell
globalThis.pgVector = require('@mastra/pg').pgVector;

// In another cell
const { pgVector } = globalThis;
```

This approach helps you avoid "not defined" errors when referencing variables or modules across different cells. It's especially useful for sharing configuration, database connections, or utility functions throughout your notebook.


---

# Table of Contents

# RAG System Components

Building RAG systems for agents with Mastra requires three components:

1. **Knowledge Store/Index** - This creates numerical representations of textual content.
2. **Retriever** - This matches the result of embedding a query with stored vectors. [More on embeddings later](#embeddings)
3. **Generator** - Creates contextually informed responses using a LLM.

# Project Setup

The easiest way to scaffold this project is by using the `npx create-mastra@latest` `bash` command, then you'll need 
to install several additional modules. These have already been installed for this project, so you don't need to run 
this code.

``````

> npx create-mastra@latest
> npm install @mastra/rag@latest @mastra/pg@latest ai@latest

## Add Keys

You need to add an API provider key and Postgres Connection Key for this lab to work.

Presently, this lab is hard coded to use `openai` as the LLM provider.

Add `OPEN_API_KEY` AND `POSTGRES_CONNECTION_STRING` to the `.env` file.

In [1]:
// const OPEN_API_KEY = Deno.env.get("OPENAI_API_KEY");
// const POSTGRES_CONNECTION_STRING = Deno.env.get("POSTGRES_CONNECTION_STRING");

const OPEN_API_KEY = 'process.env.OPENAI_API_KEY;'
const POSTGRES_CONNECTION_STRING = process.env.POSTGRES_CONNECTION_STRING;


In [2]:
OPEN_API_KEY

process.env.OPENAI_API_KEY;


# Create the Agent

The Mastra agent created in this lab will use a Vector Query Tool to perform semantic searches over a given vector store to find relevant content. To perform these actios we will need to give the agent 

1. A vector querying tool
2. A LLM to understand queries and generate responses
3. Custom instructions to guide the agent on how to analyze papers, use retrieved content, and acknowledge limitations


### Agent Code

We can begin by creating the query tool and the research agent.

In [3]:
import { createVectorQueryTool } from '@mastra/rag';
import { Agent } from '@mastra/core/agent';
import { openai } from '@ai-sdk/openai';


const vectorQueryTool = createVectorQueryTool({
    vectorStoreName: "pgVector",
    indexName: "papers",
    model: openai.embedding("text-embedding-3-small"),
});

const researchAgent = new Agent({
    name: "Research Agent",
    instructions: "Vector stores are specialized databases designed to handle and store high-dimensional vector data, which are essentially arrays of numbers representing complex data types like text, images, or audio",
    model: openai("gpt-4.1-mini"),
    tools: { vectorQueryTool }
})
globalThis.researchAgent = researchAgent;

Agent {
  component: 'AGENT',
  logger: ConsoleLogger {
    name: 'AGENT - undefined',
    level: 'error',
    transports: Map(0) {}
  },
  name: 'Research Agent',
  telemetry: undefined,
  id: 'Research Agent',
  model: OpenAIChatLanguageModel {
    specificationVersion: 'v1',
    modelId: 'gpt-4.1-mini',
    settings: {},
    config: {
      provider: 'openai.chat',
      url: [Function: url],
      headers: [Function: getHeaders],
      compatibility: 'strict',
      fetch: undefined
    }
  },
  metrics: {},
  evals: {}
}


# Create Vector Store

Vector stores are specialized databases designed to handle and store high-dimensional data, that is essentially arrays of numbers that representing complex data types like text, images, or audio. In this lab we use Postgres and [PGVector](https://github.com/pgvector/pgvector).

In [4]:
import { PgVector } from '@mastra/pg'
console.log(researchAgent)
const pgVector = new PgVector({
    // connectionString: Deno.env.get("POSTGRES_CONNECTION_STRING")
    connectionString: process.env.POSTGRES_CONNECTION_STRING
});
globalThis.pgVector = pgVector;

Agent {
  component: 'AGENT',
  logger: ConsoleLogger {
    name: 'AGENT - undefined',
    level: 'error',
    transports: Map(0) {}
  },
  name: 'Research Agent',
  telemetry: undefined,
  id: 'Research Agent',
  model: OpenAIChatLanguageModel {
    specificationVersion: 'v1',
    modelId: 'gpt-4.1-mini',
    settings: {},
    config: {
      provider: 'openai.chat',
      url: [Function: url],
      headers: [Function: getHeaders],
      compatibility: 'strict',
      fetch: undefined
    }
  },
  metrics: {},
  evals: {}
}
PgVector {
  component: 'VECTOR',
  logger: ConsoleLogger {
    name: 'VECTOR - MastraVector',
    level: 'error',
    transports: Map(0) {}
  },
  name: 'MastraVector',
  telemetry: undefined,
  pool: BoundPool {
    _events: [Object: null prototype] {},
    _eventsCount: 0,
    _maxListeners: undefined,
    options: {
      connectionString: 'your_connection_string',
      max: 20,
      idleTimeoutMillis: 30000,
      connectionTimeoutMillis: 2000,
      min: 0

# Create Mastra Instance

This lab uses a Mastra agent and gives it access to a Vector store in four lines.

In [5]:
import { Mastra } from '@mastra/core';


const mastra = new Mastra({
    agents: { researchAgent },
    vectors: { pgVector },
});


# Parse Research Article

The goal of this lab is to create an Ai agent that returns information for a given research article. We need to transform our content into a Mastra readable text format. There are several to choose from, but in this case we can use the `.fromText()` method.

In [ ]:
import { MDocument } from '@mastra/rag';
// This is a workaround to allow SSL connections to be made to the arXiv website
// It's opens a huge vulnerability. 
// Don't do this in production.
process.env.NODE_TLS_REJECT_UNAUTHORIZED = '0';

const paperURL = "https://arxiv.org/html/1706.03762";
const response = await fetch(paperURL);
const paperText = await response.text();

const doc = MDocument.fromText(paperText)
doc

(node:16111) Warning: Setting the NODE_TLS_REJECT_UNAUTHORIZED environment variable to '0' makes TLS connections and HTTPS requests insecure by disabling certificate verification.
(Use `node --trace-warnings ...` to show where the warning was created)


_MDocument {
  chunks: [
    Document {
      id_: '09d37833-443d-428f-a7cb-aff5f2c61f88',
      metadata: {},
      relationships: {},
      text: '<!DOCTYPE html>\n' +
        '<html lang="en">\n' +
        '<head>\n' +
        '<meta content="text/html; charset=utf-8" http-equiv="content-type"/>\n' +
        '<title>Attention Is All You Need</title>\n' +
        '<!--Generated on Tue Apr 30 16:03:52 2024 by LaTeXML (version 0.8.8) http://dlmf.nist.gov/LaTeXML/.-->\n' +
        '<meta content="width=device-width, initial-scale=1, shrink-to-fit=no" name="viewport"/>\n' +
        '<link href="https://cdn.jsdelivr.net/npm/bootstrap@5.3.0/dist/css/bootstrap.min.css" rel="stylesheet" type="text/css"/>\n' +
        '<link href="/static/browse/0.3.4/css/ar5iv.0.7.9.min.css" rel="stylesheet" type="text/css"/>\n' +
        '<link href="/static/browse/0.3.4/css/ar5iv-fonts.0.7.9.min.css" rel="stylesheet" type="text/css"/>\n' +
        '<link href="/static/browse/0.3.4/css/latexml_styles.css" r

In [ ]:
const { diagnoseSSLIssues } = require('../src/config/ssl');
await diagnoseSSLIssues('https://arxiv.org/html/1706.03762');